In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import seaborn as sns
#sns.set(style='whitegrid')
sns.set(style='whitegrid')

dataset = 'fineweb' # openwebtext or shakespeare
scion_methods = ['constrained_scion','unconstrained_scion'] # unconstrained_scion or constrained_scion or projected_scion
loss_datasets = ['train', 'val'] # train or val
alphas = [0.5, 1.0]
epochs=5100
out_folder="out"


In [ ]:
METHOD_MAP = {
    'unconstrained_scion': 'Unconstrained Scion',
    'projected_scion': 'Projected Scion',
    'constrained_scion': 'Scion'
}

widths = [256]
depths = [2, 4, 8, 16, 32]
lrs = [0.001953125, 0.0009765625, 0.00048828125, 0.000244140625, 0.0001220703125, 6.103515625e-05]
        
for alpha in alphas:
    for loss_dataset in loss_datasets:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axs = axes.ravel()
        
        for i, scion_method in enumerate(scion_methods):
            parameterization_str = METHOD_MAP.get(scion_method, scion_method)
            extra = 'Spectral'
            ax = axs[i]

            optimal_lrs = []
            optimal_losses = []
            palette = sns.color_palette("viridis", n_colors=len(depths))

            for depth_idx, depth in enumerate(depths):
                seeds = [1,2,3]
                
                mean_losses = []
                sem_losses = []
                lrs_to_plot = []

                for lr in lrs:
                    losses = []
                    for seed in seeds:
                        for width in widths:
                            if scion_method == 'scion_full':
                                job_name = f'mode{extra}_width{width}_depth2_seed{seed}_lr{lr:.20f}'.rstrip('0')
                            else:
                                job_name = f'first_layer{extra}_width{width}_depth{depth}_seed{seed}_lr{lr:.20f}'.rstrip('0')

                            csv_path = os.path.join(
                                f'depthtransfer_{dataset}/{out_folder}/{scion_method}_out', 
                                f'hptransferdepth_alpha_skipconnection_{alpha}_spectral_sign__depth{depth}_seed{seed}_lr{lr}', 
                                'log.csv'
                            )

                            if os.path.exists(csv_path):
                                ckpt_df = pd.read_csv(csv_path, header=1, skip_blank_lines=True)
                                losses.append(ckpt_df[f'{loss_dataset}/loss'].values[-1])
                                # losses.append(ckpt_df[f'{loss_dataset}/loss'].ewm(alpha=0.8).mean().values[-1]) # CompleteP
                                # losses.append(ckpt_df[f'{loss_dataset}/loss'].min())
                            else:
                                print(f'Missing {csv_path}')

                    if losses:
                        mean_val = np.mean(losses)
                        mean_losses.append(mean_val)
                        lrs_to_plot.append(lr)
                        if len(seeds) > 1:
                            sem_losses.append(np.std(losses, ddof=1) / np.sqrt(len(losses)))

                # Plots
                if mean_losses:
                    color = palette[len(depths) - depth_idx - 1]
                    ax.plot(lrs_to_plot, mean_losses, label=depth, color=color, marker='o', markersize=4)
                    
                    if len(seeds) > 1 and sem_losses:
                        ax.fill_between(lrs_to_plot, 
                                        np.array(mean_losses) - np.array(sem_losses), 
                                        np.array(mean_losses) + np.array(sem_losses), 
                                        color=color, alpha=0.33)

                    optimum_idx = np.argmin(mean_losses)
                    optimal_lrs.append(lrs_to_plot[optimum_idx])
                    optimal_losses.append(mean_losses[optimum_idx])

            ax.plot(optimal_lrs, optimal_losses, color='red', linestyle='none', marker='o')
            ax.set_xscale('log', base=2)
            ax.set_xlabel('Base learning rate')
            ax.set_title(f"{parameterization_str} with " + r"$\alpha =$" + f"{alpha}")
            if i==0:
                ax.legend(title='Depth', loc='upper right')

            if i == 0:
                ax.set_ylabel(f'{"Validation" if loss_dataset=="val" else "Train"} Loss')
            else:
                ax.yaxis.set_ticklabels([])
                ax.tick_params(axis='y', length=0)

        plt.tight_layout()
        # plt.savefig(f"GPT_{dataset}_{out_folder}depth_transfer_scion_and_uscion_alpha_{alpha}_{loss_dataset}_loss_{epochs}_epochs_{len(seeds)}_seeds.pdf")
        plt.show()
        plt.close()